In [6]:
import torch
from torch import nn
import numpy as np
import gym
import torch.nn.functional as F

setattr(np, "bool8", np.bool)

In [7]:
class Actor(nn.Module):
    def __init__(self, obs_n=4, act_n=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_n, 128),
            nn.ReLU(),
            nn.Linear(128, act_n)
        )
    
    def forward(self, obs):
        return self.net(obs)

class Critor(nn.Module):
    def __init__(self, obs_n=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_n, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    
    def forward(self, obs):
        return self.net(obs)

In [8]:
class ActorCritor:
    def __init__(self, obs_n=4, act_n=2, actor_lr=1e-3, critor_lr=1e-2,gamma=0.98, device="cpu"):
        self.device = device
        self.actor = Actor(obs_n, act_n).to(self.device)
        self.critor = Critor(obs_n).to(self.device)
        self.optimizer_actor = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.optimizer_critor = torch.optim.Adam(self.critor.parameters(), lr=critor_lr)
        self.gamma = gamma
        
    
    def take_action(self, obs):
        obs = torch.tensor([obs], dtype=torch.float).to(self.device)
        logits = self.actor(obs)
        probs = torch.distributions.Categorical(logits=logits)
        action = probs.sample()
        return action.item()
    
    def learn(self, obs_list, action_list, reward_list, next_obs_list, done_list):
        obs = torch.tensor(obs_list, dtype=torch.float).to(self.device)
        action = torch.tensor(action_list, dtype=torch.long).to(self.device)
        reward = torch.tensor(reward_list, dtype=torch.float).to(self.device).reshape(-1, 1)
        next_obs = torch.tensor(next_obs_list, dtype=torch.float).to(self.device)
        done = torch.tensor(done_list, dtype=torch.float).to(self.device).reshape(-1, 1)


        td_target = reward + self.gamma * self.critor(next_obs) * (1 - done)
        td_error = td_target - self.critor(obs)

        ce_loss = F.cross_entropy(self.actor(obs), action, reduction="none")
        actor_loss = torch.mean(ce_loss*td_error.detach().squeeze(1))


        critor_loss = F.mse_loss(td_target.detach(), self.critor(obs))

        self.optimizer_actor.zero_grad()
        actor_loss.backward()
        self.optimizer_actor.step()

        self.optimizer_critor.zero_grad()
        critor_loss.backward()
        self.optimizer_critor.step()



In [12]:
env = gym.make("CartPole-v0")

episode = 1000

actor_lr = 1e-3
critor_lr = 1e-2
gamma = 0.98
device = "cpu"
agent = ActorCritor(
    obs_n=4,
    act_n=2,
    actor_lr=actor_lr,
    critor_lr=critor_lr,
    gamma=gamma,
    device=device,
)

for i in range(episode):
    obs_list , action_list, reward_list, next_obs_list, done_list = [], [], [], [], []
    obs = env.reset()[0]
    while True:
        action = agent.take_action(obs)
        next_observation, reward, done, truncated, info = env.step(action)

        obs_list.append(obs)
        action_list.append(action)
        reward_list.append(reward)
        next_obs_list.append(next_observation)
        done_list.append(done)

        obs = next_observation
        if done:
            break
    
    agent.learn(obs_list , action_list, reward_list, next_obs_list, done_list)
    print(f"Episode: {i+1}, Reward:{sum(reward_list)}")
    

Episode: 1, Reward:18.0
Episode: 2, Reward:11.0
Episode: 3, Reward:19.0
Episode: 4, Reward:15.0
Episode: 5, Reward:21.0
Episode: 6, Reward:17.0
Episode: 7, Reward:23.0
Episode: 8, Reward:17.0
Episode: 9, Reward:20.0
Episode: 10, Reward:11.0
Episode: 11, Reward:9.0
Episode: 12, Reward:19.0
Episode: 13, Reward:9.0
Episode: 14, Reward:31.0
Episode: 15, Reward:13.0
Episode: 16, Reward:17.0
Episode: 17, Reward:15.0
Episode: 18, Reward:11.0
Episode: 19, Reward:17.0
Episode: 20, Reward:13.0
Episode: 21, Reward:16.0
Episode: 22, Reward:59.0
Episode: 23, Reward:16.0
Episode: 24, Reward:10.0
Episode: 25, Reward:34.0
Episode: 26, Reward:16.0
Episode: 27, Reward:19.0
Episode: 28, Reward:18.0
Episode: 29, Reward:22.0
Episode: 30, Reward:12.0
Episode: 31, Reward:13.0
Episode: 32, Reward:22.0
Episode: 33, Reward:25.0
Episode: 34, Reward:13.0
Episode: 35, Reward:16.0
Episode: 36, Reward:8.0
Episode: 37, Reward:14.0
Episode: 38, Reward:15.0
Episode: 39, Reward:15.0
Episode: 40, Reward:18.0
Episode: 41,